# Supply-Shock Data Pipeline — Starter Notebook

**Goal:** gather the key data from open sources, normalize it onto three join keys
(**country codes · dates · coordinates**), join it into one table, and use that table
to *compose a story* — trace an event (e.g. a chokepoint disruption) through to the
airfields it affects.

**Sources in this starter (the open-API core):**

| Source | What it gives | Access |
|---|---|---|
| OurAirports | 80k+ airfields with coordinates | public-domain CSV |
| JODI-Oil | production / stocks by country | CSV download |
| World Bank | economic exposure (GDP, energy) | `wbgapi` API |
| UN Comtrade | trade flows (who imports crude) | API (free tier) |
| EIA | prices, stocks, crude imports | REST API (free key) |

> This is deliberately the *reliable, no-scraping* set. GDELT/ACLED (events),
> IMF PortWatch (chokepoint transits), and OPEC MOMR (PDF tables) are added as
> **Extensions** at the bottom.

**Principle:** you don't scrape these — you use APIs and downloads. Keep API keys in
environment variables, respect rate limits, and mind licensing (e.g. ACLED is
non-commercial only).


## 0 · Setup

Install once (uncomment), then import. Put your API keys in a `.env` file or your shell
environment — never hardcode them.


In [59]:
# Dependencies already installed. Uncomment and re-run only if you need to reinstall.
# %pip install -q pandas requests wbgapi comtradeapicall pycountry python-dotenv pyarrow openpyxl xlrd

import os, io, time, requests
import pandas as pd
from pathlib import Path

try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass

CACHE = Path("data_cache"); CACHE.mkdir(exist_ok=True)

# Keys (set these in your environment / .env):
EIA_API_KEY  = os.environ.get("EIA_API_KEY", "")
COMTRADE_KEY = os.environ.get("COMTRADE_KEY", "")

# ── Per-source cache TTLs (hours) ─────────────────────────────────────────────
# Set to match each source's actual publication cadence, not a universal default.
CACHE_TTL = {
    "gdelt_discovery":    6,      # updates every 15 min; we refresh every 6h
    "eia_bulk":          24,      # daily prices/stocks
    "portwatch":         24,      # daily transit counts
    "eia_stocks":       168,      # weekly (7 days)
    "jodi":             720,      # monthly (~30 days); also has ~2 month publication lag
    "gpr":              720,      # monthly
    "pink_prices":      720,      # monthly
    "comtrade":         720,      # monthly
    "worldbank":       8760,      # annual (365 days)
    "airports":        2160,      # rarely changes; refresh every 90 days
}
DEFAULT_TTL = 24   # fallback for any source not listed above


def cached(name, fetch_fn, max_age_hours=None):
    """
    Fetch via fetch_fn() and cache to parquet.
    Uses per-source TTL from CACHE_TTL if max_age_hours is not explicitly passed.
    Warns if data is older than 2x its TTL.
    """
    ttl   = max_age_hours if max_age_hours is not None else CACHE_TTL.get(name, DEFAULT_TTL)
    fp    = CACHE / f"{name}.parquet"
    now   = time.time()

    if fp.exists():
        age_hours = (now - fp.stat().st_mtime) / 3600
        if age_hours < ttl:
            return pd.read_parquet(fp)
        if age_hours > ttl * 2:
            print(f"  [STALE] {name}: last fetched {age_hours:.0f}h ago "
                  f"(TTL {ttl}h) — data may be significantly out of date.")

    df = fetch_fn()
    try:
        df.to_parquet(fp)
    except Exception as e:
        print(f"  [cache] could not write {fp.name} ({e}); returning uncached data.")
    return df


def cache_status():
    """
    Print a freshness summary for all cached sources:
    when last fetched, age vs TTL, and latest date in the data (if a date column exists).
    """
    print("=" * 72)
    print("CACHE STATUS")
    print("  {:<22}  {:<12}  {:<8}  {:<8}  {}".format(
          "Source", "Last fetched", "Age", "TTL", "Latest data date"))
    print("  {}  {}  {}  {}  {}".format("-"*22, "-"*12, "-"*8, "-"*8, "-"*18))

    for name in sorted(CACHE_TTL.keys()):
        fp  = CACHE / f"{name}.parquet"
        ttl = CACHE_TTL[name]
        if not fp.exists():
            print("  {:<22}  {:<12}  {:<8}  {:<8}  {}".format(
                  name, "not cached", "-", str(ttl)+"h", "-"))
            continue

        age_h   = (time.time() - fp.stat().st_mtime) / 3600
        age_str = "{:.0f}h ago".format(age_h)
        stale   = " !" if age_h > ttl * 2 else ""

        latest_date = "-"
        try:
            df = pd.read_parquet(fp)
            for col in ["date", "seendate", "TIME_PERIOD"]:
                if col in df.columns:
                    val = pd.to_datetime(df[col], errors="coerce").max()
                    if pd.notna(val):
                        latest_date = str(val.date())
                        break
        except Exception:
            pass

        print("  {:<22}  {:<12}  {:<8}  {:<8}  {}{}".format(
              name,
              pd.Timestamp(fp.stat().st_mtime, unit="s").strftime("%Y-%m-%d"),
              age_str,
              str(ttl) + "h",
              latest_date,
              stale))

    print("  ! = older than 2x TTL")
    print("=" * 72)

cache_status()


CACHE STATUS
  Source                  Last fetched  Age       TTL       Latest data date
  ----------------------  ------------  --------  --------  ------------------
  airports                2026-07-21    7h ago    2160h     -
  comtrade                not cached    -         720h      -
  eia_bulk                2026-07-21    6h ago    24h       2026-07-13
  eia_stocks              not cached    -         168h      -
  gdelt_discovery         2026-07-21    4h ago    6h        2026-07-21
  gpr                     2026-07-21    7h ago    720h      2026-06-01
  jodi                    not cached    -         720h      -
  pink_prices             2026-07-21    7h ago    720h      2026-06-01
  portwatch               2026-07-21    7h ago    24h       2026-07-19
  worldbank               2026-07-21    7h ago    8760h     -
  ! = older than 2x TTL


## 1 · OurAirports — the base map (coordinates)

Public-domain CSV, no key. This is the board every other layer is overlaid on.
We keep the busier airports and the fields we need to join and locate them.


In [60]:
AIRPORTS_URL = "https://davidmegginson.github.io/ourairports-data/airports.csv"

def load_airports():
    df = pd.read_csv(AIRPORTS_URL)
    keep = ["ident","type","name","iso_country","latitude_deg","longitude_deg","iata_code"]
    df = df[keep]
    # focus on airports that actually take jet traffic
    df = df[df["type"].isin(["large_airport","medium_airport"])]
    return df.reset_index(drop=True)

airports = cached("airports", load_airports)
print(f"{len(airports):,} airports")
airports.head()


5,276 airports


,ident,type,name,iso_country,latitude_deg,longitude_deg,iata_code
0,07FA,medium_airport,Ocean Reef Club Airport,US,25.325399,-80.274803,OCA
1,5A8,medium_airport,Aleknagik / New Airport,US,59.282600,-158.617996,WKK
2,AE-0221,medium_airport,Sas Al Nakheel Air Base,AE,24.441280,54.516960,None
3,AG-0001,medium_airport,Burton-Nibbs International Airport,AG,17.621194,-61.798347,BBQ
4,AGGH,large_airport,Honiara International Airport,SB,-9.428000,160.054993,HIR


## 2 · JODI-Oil — production & stocks by country (join key: country)

JODI publishes the **World Database** as a CSV. Download the latest from
<https://www.jodidata.org/oil/database/data-downloads.aspx> and point `JODI_CSV` at the
file (it may sit behind a click, so a local path is most reliable). Expected schema below.


In [61]:
# After downloading, set this to your local file:
JODI_CSV = CACHE / "jodi_oil_world.csv"     # <-- put the downloaded CSV here

# JODI columns: REF_AREA (ISO2 country), ENERGY_PRODUCT, FLOW_BREAKDOWN,
#               UNIT_MEASURE, TIME_PERIOD (YYYY-MM), OBS_VALUE, ASSESSMENT_CODE
#
# NOTE: JODI reports every (country, month) in FIVE units (CONVBBL, KBBL, KBD, KL,
# KTONS). We MUST pin one unit, or a later groupby will mix them and the numbers
# become meaningless. KBD = thousand barrels/day, the standard production metric.
JODI_UNIT = "KBD"

def load_jodi():
    if not JODI_CSV.exists():
        print("JODI file not found — download it and set JODI_CSV. Skipping for now.")
        return pd.DataFrame(columns=["REF_AREA","ENERGY_PRODUCT","FLOW_BREAKDOWN",
                                     "TIME_PERIOD","OBS_VALUE"])
    df = pd.read_csv(JODI_CSV)
    # crude oil production (CRUDEOIL / INDPROD) in a single unit is the headline supply series
    m = ((df["ENERGY_PRODUCT"]=="CRUDEOIL") &
         (df["FLOW_BREAKDOWN"]=="INDPROD") &
         (df["UNIT_MEASURE"]==JODI_UNIT))
    out = df[m].copy()
    out["OBS_VALUE"] = pd.to_numeric(out["OBS_VALUE"], errors="coerce")  # "-" placeholders -> NaN
    return out

jodi = load_jodi()
jodi.head()


,REF_AREA,TIME_PERIOD,ENERGY_PRODUCT,FLOW_BREAKDOWN,UNIT_MEASURE,OBS_VALUE,ASSESSMENT_CODE
12,AE,2002-01,CRUDEOIL,INDPROD,KBD,1955.0,1
62,AE,2002-02,CRUDEOIL,INDPROD,KBD,1941.0,1
112,AE,2002-03,CRUDEOIL,INDPROD,KBD,1921.0,1
162,AE,2002-04,CRUDEOIL,INDPROD,KBD,1769.0,2
212,AE,2002-05,CRUDEOIL,INDPROD,KBD,1894.0,1


## 3 · World Bank — economic exposure (join key: country)

The `wbgapi` package needs no key. Pull indicators that tell you which economies are most
exposed to an oil-supply shock (GDP, oil rents, energy use).


In [62]:
import wbgapi as wb

INDICATORS = {
    "NY.GDP.MKTP.CD":  "gdp_usd",
    "NY.GDP.PETR.RT.ZS":"oil_rents_pct_gdp",
    "EG.USE.COMM.KT.OE":"energy_use_ktoe",
}

def load_worldbank():
    df = wb.data.DataFrame(list(INDICATORS), mrv=1)   # most-recent value
    df = df.rename(columns=INDICATORS).reset_index().rename(columns={"economy":"iso3"})
    return df

worldbank = cached("worldbank", load_worldbank)
worldbank.head()


,iso3,gdp_usd,oil_rents_pct_gdp
0,ABW,NaN,NaN
1,AFE,1.358694e+12,NaN
2,AFG,NaN,NaN
3,AFW,8.525270e+11,NaN
4,AGO,1.221749e+11,NaN


## 4 · UN Comtrade — who imports crude (join key: country)

Trade flows show the downstream exposure — which countries buy crude, and from whom.
Commodity code **2709** = crude petroleum oils. The free preview endpoint works without a
key (limited rows); a subscription key raises the limits.


In [63]:
import comtradeapicall
import datetime as _dt_ct

def _comtrade_period():
    """Most recent month Comtrade is likely to have published (2-3 month lag)."""
    y, m = _dt_ct.date.today().year, _dt_ct.date.today().month
    m -= 3
    if m <= 0:
        m += 12
        y -= 1
    return "{:04d}{:02d}".format(y, m)

def load_crude_imports(period=None):
    if period is None:
        period = _comtrade_period()
    print("  Comtrade: requesting period {} (HS 2709 = crude petroleum)...".format(period))
    try:
        df = comtradeapicall.previewFinalData(
            typeCode="C", freqCode="M", clCode="HS", period=period,
            reporterCode=None, cmdCode="2709", flowCode="M",
            partnerCode=None, partner2Code=None,
            customsCode=None, motCode=None, maxRecords=2500, format_output="JSON",
            countOnly=None, includeDesc=True,
        )
        if df is not None and not df.empty:
            print("  Comtrade: {:,} rows loaded for period {}".format(len(df), period))
        return df if df is not None else pd.DataFrame()
    except Exception as e:
        print("Comtrade preview failed (rate limit or network):", e)
        return pd.DataFrame()

# Cache key includes period so stale data is not silently reused when the month rolls over.
_ct_period = _comtrade_period()
crude_imports = cached("comtrade_" + _ct_period, lambda: load_crude_imports(_ct_period))
crude_imports.head()


  Comtrade: requesting period 202604 (HS 2709 = crude petroleum)...
  Comtrade: 500 rows loaded for period 202604


,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,M,20260401,2026,4,202604,31,AZE,Azerbaijan,M,...,22345133.0,False,0.000,False,1.667737e+07,NaN,1.667737e+07,0,False,True
1,C,M,20260401,2026,4,202604,31,AZE,Azerbaijan,M,...,22345133.0,False,0.000,False,1.667737e+07,NaN,1.667737e+07,0,False,True
2,C,M,20260401,2026,4,202604,36,AUS,Australia,M,...,798227801.0,False,649044.111,False,6.760885e+08,6.437813e+08,6.760885e+08,0,False,True
3,C,M,20260401,2026,4,202604,36,AUS,Australia,M,...,145663031.0,False,116817.818,False,1.252029e+08,1.179866e+08,1.252029e+08,0,False,True
4,C,M,20260401,2026,4,202604,36,AUS,Australia,M,...,189670556.0,False,153880.865,False,1.514154e+08,1.438682e+08,1.514154e+08,0,False,True


## 5 · EIA — prices, stocks & crude imports (join key: date)

EIA offers the same data two ways: a keyed REST API, or **public bulk downloads that need
no key at all**. We use the **bulk** route — whole datasets ship as one zip of
newline-delimited JSON — so nothing here requires signing up. Attribute "U.S. EIA".

We pull a few headline series from the `PET` (Petroleum) dataset: WTI & Brent spot prices,
US crude imports, and crude stocks. Swap in other `series_id`s (e.g. from the `STEO`
forecast dataset) by editing `EIA_BULK`.


In [64]:
# EIA — NO API KEY NEEDED: use the public bulk downloads, not the keyed REST API.
# A whole dataset ships as one zip of newline-delimited JSON. Attribute "U.S. EIA".
import zipfile, json as _json

EIA_BULK = {   # EIA series_id -> friendly metric name   (all from the PET dataset)
    "PET.RWTC.D":                 "wti_usd_bbl",          # WTI spot price, daily
    "PET.RBRTE.D":                "brent_usd_bbl",        # Brent spot price, daily
    "PET.EER_EPJK_PF4_RGC_DPG.D": "jet_fuel_usd_gal",     # US Gulf Coast jet-fuel spot, daily (layer 4)
    "PET.MCRIMUS2.M":             "us_crude_imports_kbd", # US crude oil imports, monthly
    "PET.WCESTUS1.W":             "us_crude_stocks_kbbl", # US crude oil stocks, weekly (verify gate)
}

def _eia_bulk_zip(dataset="PET", max_age_hours=24):
    """Download an EIA bulk dataset zip (no key) into the cache, reusing if fresh."""
    fp = CACHE / f"{dataset}.zip"
    if not (fp.exists() and (time.time() - fp.stat().st_mtime) < max_age_hours*3600):
        r = requests.get(f"https://api.eia.gov/bulk/{dataset}.zip", timeout=180); r.raise_for_status()
        fp.write_bytes(r.content)
    return fp

def load_eia_bulk(series=EIA_BULK, dataset="PET"):
    fp = _eia_bulk_zip(dataset)
    want = set(series); rows = []
    with zipfile.ZipFile(fp) as z:
        inner = [n for n in z.namelist() if n.endswith(".txt")][0]
        with z.open(inner) as fh:
            for raw in io.TextIOWrapper(fh, encoding="utf-8"):
                if not want: break                              # stop once every series is found
                if not any(s in raw for s in want): continue    # cheap prefilter before json.loads
                o = _json.loads(raw); sid = o.get("series_id")
                if sid in want and o.get("data"):
                    rows += [(series[sid], p, v) for p, v in o["data"]]
                    want.discard(sid)
    df = pd.DataFrame(rows, columns=["metric", "period", "value"])
    p = df["period"].astype(str)                # periods are mixed: YYYYMMDD / YYYYMM / YYYY
    df["date"] = pd.NaT
    for L, fmt in [(8, "%Y%m%d"), (6, "%Y%m"), (4, "%Y")]:
        m = p.str.len() == L
        df.loc[m, "date"] = pd.to_datetime(p[m], format=fmt, errors="coerce")
    return df

eia = cached("eia_bulk", load_eia_bulk)
if not eia.empty:
    latest = (eia.sort_values("date").groupby("metric")
                 .agg(latest_date=("date","last"), latest_value=("value","last")))
    print("EIA (no key) — latest values:"); print(latest.to_string())
eia.head()


EIA (no key) — latest values:
                     latest_date  latest_value
metric                                        
brent_usd_bbl         2026-07-13        81.620
jet_fuel_usd_gal      2026-07-13         3.379
us_crude_imports_kbd  2026-04-01      6257.000
us_crude_stocks_kbbl  2026-07-10    409665.000
wti_usd_bbl           2026-07-13        79.200


,metric,period,value,date
0,wti_usd_bbl,20260713,79.20,2026-07-13
1,wti_usd_bbl,20260710,72.45,2026-07-10
2,wti_usd_bbl,20260709,73.15,2026-07-09
3,wti_usd_bbl,20260708,74.56,2026-07-08
4,wti_usd_bbl,20260707,71.53,2026-07-07


## 5b · Commercial-safe extension sources (no API key)

Four more sources that load with **no key** and are cleared for **commercial use** (just
attribute each). They fill the price, chokepoint-transit, geopolitical-risk, and news
layers of the story.

| Source | Layer it fills | License / attribution |
|---|---|---|
| World Bank Pink Sheet | crude & commodity **prices** | CC BY 4.0 — "World Bank Commodity Markets" |
| IMF PortWatch | **chokepoint** transit volumes | free w/ attribution — "IMF PortWatch" |
| GPR Index (Caldara–Iacoviello) | geopolitical-**risk** premium | free — cite Caldara & Iacoviello (2022) |
| GDELT 2.0 | global **news/event** signal | open — "The GDELT Project" |

> **ACLED is deliberately excluded.** Its free data is *non-commercial only*, so it can't
> go in a commercial product without a paid ACLED license. Everything above can.


In [65]:
# World Bank "Pink Sheet" — monthly commodity prices incl. crude (Brent/Dubai/WTI). CC BY 4.0.
# NOTE: the dated path segment (…-0050012026/…) changes when the workbook is refreshed.
# If this 404s, grab the current .xlsx link from worldbank.org/en/research/commodity-markets.
PINK_URL = ("https://thedocs.worldbank.org/en/doc/"
            "74e8be41ceb20fa0da750cda2f6b9e4e-0050012026/related/CMO-Historical-Data-Monthly.xlsx")

def load_pink_prices():
    raw = pd.read_excel(PINK_URL, sheet_name="Monthly Prices", header=None)
    names = raw.iloc[4].tolist()               # row 4 holds the commodity names
    data = raw.iloc[6:].copy()                 # data rows start at row 6
    data.columns = ["period"] + list(names[1:])
    crude = [c for c in data.columns if isinstance(c, str) and c.startswith("Crude oil")]
    out = data[["period"] + crude].copy()
    out["date"] = pd.to_datetime(out["period"].astype(str).str.replace("M", "-", regex=False),
                                 format="%Y-%m", errors="coerce")
    for c in crude:
        out[c] = pd.to_numeric(out[c], errors="coerce")   # "…" placeholders -> NaN
    return out.dropna(subset=["date"]).drop(columns=["period"]).reset_index(drop=True)

pink_prices = cached("pink_prices", load_pink_prices)
if not pink_prices.empty:
    print(f"{len(pink_prices):,} months of prices, through {pink_prices['date'].max().date()}")
pink_prices.tail(3)


798 months of prices, through 2026-06-01


,"Crude oil, average","Crude oil, Brent","Crude oil, Dubai","Crude oil, WTI",date
795,103.9,120.4,92.7,98.6,2026-04-01
796,100.4,107.5,94.7,99.1,2026-05-01
797,81.7,85.4,77.7,81.9,2026-06-01


In [66]:
# IMF PortWatch — daily transit counts & capacity at maritime chokepoints. Attribute "IMF PortWatch."
PW_CHOKE_URL = ("https://services9.arcgis.com/weJ1QsnbMYJlCHdG/arcgis/rest/services/"
                "Daily_Chokepoints_Data/FeatureServer/0/query")
# this notebook's short chokepoint names -> PortWatch "portname" values
PW_NAME = {"Hormuz":"Strait of Hormuz", "Suez":"Suez Canal", "Malacca":"Malacca Strait",
           "BabelMandeb":"Bab el-Mandeb Strait", "Panama":"Panama Canal"}

def load_portwatch():
    names = "','".join(PW_NAME.values())
    params = {"where": f"portname IN ('{names}')",
              "outFields": "date,portname,n_tanker,n_cargo,n_total,capacity_tanker,capacity",
              "orderByFields": "date DESC",       # newest first; server caps at 1000 rows
              "resultRecordCount": 1000, "f": "json"}
    r = requests.get(PW_CHOKE_URL, params=params, timeout=40); r.raise_for_status()
    df = pd.DataFrame([f["attributes"] for f in r.json().get("features", [])])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df

portwatch = cached("portwatch", load_portwatch)
if not portwatch.empty:
    print(f"{len(portwatch):,} chokepoint-day rows, {portwatch['portname'].nunique()} chokepoints, "
          f"through {portwatch['date'].max().date()}")
portwatch.head()


1,000 chokepoint-day rows, 5 chokepoints, through 2026-07-19


,date,portname,n_tanker,n_cargo,n_total,capacity_tanker,capacity
0,2026-07-19,Suez Canal,21,22,43,1443001,1973255
1,2026-07-19,Panama Canal,18,16,34,449996,873923
2,2026-07-19,Bab el-Mandeb Strait,18,15,33,1031074,1341101
3,2026-07-19,Malacca Strait,65,105,170,3647281,7672211
4,2026-07-19,Strait of Hormuz,4,11,15,39216,182620


In [67]:
# GPR geopolitical-risk index (Caldara & Iacoviello, 2022). Free — cite the authors.
GPR_URL = "https://www.matteoiacoviello.com/gpr_files/data_gpr_export.xls"

def load_gpr():
    g = pd.read_excel(GPR_URL, sheet_name="Sheet1").rename(columns={"month": "date"})
    keep = [c for c in ["date","GPR","GPRT","GPRA"] if c in g.columns]   # index, threats, acts
    g = g[keep].dropna(subset=["GPR"]).copy()
    g["date"] = pd.to_datetime(g["date"], errors="coerce")
    return g.dropna(subset=["date"]).reset_index(drop=True)

gpr = cached("gpr", load_gpr)
if not gpr.empty:
    print(f"{len(gpr):,} months; latest GPR {gpr['GPR'].iloc[-1]:.1f} ({gpr['date'].iloc[-1].date()})")
gpr.tail(3)


498 months; latest GPR 173.6 (2026-06-01)


,date,GPR,GPRT,GPRA
495,2026-04-01,254.429504,274.106720,310.083588
496,2026-05-01,203.556671,233.463852,215.219376
497,2026-06-01,173.639557,210.581039,177.959946


In [68]:
# GDELT 2.0 DOC API — broad energy-disruption discovery query. No key. Attribute "The GDELT Project."
# Replaces the fixed chokepoint query with a broad signal that surfaces any oil-relevant
# event without pre-specifying geography or event type.

GDELT_DISCOVERY_QUERY = (
    '(oil OR crude OR petroleum OR LNG OR tanker OR pipeline) '
    'AND (ceasefire OR sanctions OR attack OR conflict OR disruption '
    'OR blockade OR embargo OR escalation OR explosion OR shutdown)'
)

def load_gdelt_discovery(query=GDELT_DISCOVERY_QUERY, days=90, maxrecords=250):
    """
    Broad GDELT query for event discovery.
    Tries a 90-day window first; falls back to 30 days if the API rejects the longer span.
    Returns articles with seendate, title, sourcecountry.
    """
    url = "https://api.gdeltproject.org/api/v2/doc/doc"
    for timespan in [f"{days}d", "30d"]:
        params = {
            "query":      query,
            "mode":       "artlist",
            "format":     "json",
            "maxrecords": maxrecords,
            "timespan":   timespan,
            "sort":       "datedesc",
        }
        for attempt in range(4):
            r = requests.get(url, params=params, timeout=40)
            if r.status_code == 200:
                try:
                    arts = r.json().get("articles", [])
                except ValueError:
                    arts = None
                if arts is not None:
                    df = pd.DataFrame(arts)
                    if not df.empty:
                        df["seendate"] = pd.to_datetime(
                            df["seendate"], format="%Y%m%dT%H%M%SZ", errors="coerce"
                        )
                    print(f"  GDELT: {len(df):,} articles retrieved ({timespan} window)")
                    return df
            time.sleep(2 * (attempt + 1))
        print(f"  GDELT: {timespan} window failed, trying shorter span...")
    print("GDELT unavailable after retries.")
    return pd.DataFrame()

gdelt_discovery = cached("gdelt_discovery", load_gdelt_discovery, max_age_hours=6)


## 6 · Normalize onto the join keys

Everything gets a consistent **ISO-3 country code**, parsed **dates**, and (for airports)
**coordinates**. `pycountry` maps names/ISO2 → ISO3.


In [69]:
import pycountry

def to_iso3(x):
    if pd.isna(x): return None
    x = str(x).strip()
    if not x: return None
    try:
        if x.isdigit():                                    # Comtrade/M49 numeric code, e.g. "842" -> USA
            return pycountry.countries.get(numeric=x.zfill(3)).alpha_3
        if len(x)==2:  return pycountry.countries.get(alpha_2=x).alpha_3
        if len(x)==3:  return pycountry.countries.get(alpha_3=x).alpha_3
        return pycountry.countries.lookup(x).alpha_3       # fall back to a name lookup
    except Exception:
        return None                                        # aggregates like World ("000") land here

airports["iso3"] = airports["iso_country"].map(to_iso3)
if not jodi.empty:
    jodi["iso3"] = jodi["REF_AREA"].map(to_iso3)

if not crude_imports.empty:
    # reporter = the importing country; partner = who it bought the crude from
    rep = crude_imports["reporterISO"] if "reporterISO" in crude_imports else crude_imports.get("reporterCode")
    crude_imports["iso3"] = rep.map(to_iso3) if rep is not None else None
    par = crude_imports["partnerISO"] if "partnerISO" in crude_imports else crude_imports.get("partnerCode")
    crude_imports["partner_iso3"] = par.map(to_iso3) if par is not None else None
    print("comtrade rows with reporter iso3:", crude_imports["iso3"].notna().mean().round(2),
          "| with partner iso3:", crude_imports["partner_iso3"].notna().mean().round(2))

print("normalized. airports with iso3:", airports["iso3"].notna().mean().round(2))


comtrade rows with reporter iso3: 1.0 | with partner iso3: 0.79
normalized. airports with iso3: 1.0


## 7 · Assemble one country-level table

Merge the country-keyed sources (World Bank exposure, JODI production, Comtrade imports)
into a single frame. Airports stay in their own table, joined to countries by `iso3`
(and locatable by coordinates).


In [70]:
country = worldbank.copy() if not worldbank.empty else pd.DataFrame({"iso3":[]})

if not jodi.empty:
    prod = (jodi.sort_values("TIME_PERIOD").groupby("iso3")["OBS_VALUE"].last()
                .rename("crude_prod_latest").reset_index())
    country = country.merge(prod, on="iso3", how="outer")

if not crude_imports.empty and "iso3" in crude_imports:
    # Sum only bilateral rows (partner_iso3 present). Comtrade also returns a per-reporter
    # "World" total row whose partner won't map to an iso3 — including it would double-count.
    bilateral = crude_imports[crude_imports["partner_iso3"].notna()]
    imp = (bilateral.groupby("iso3")["primaryValue"].sum()
               .rename("crude_import_value").reset_index())
    country = country.merge(imp, on="iso3", how="outer")

country.to_parquet(CACHE / "country_assembled.parquet")
print(country.shape); country.head()


(266, 5)


,iso3,gdp_usd,oil_rents_pct_gdp,crude_prod_latest,crude_import_value
0,ABW,NaN,NaN,NaN,NaN
1,AFE,1.358694e+12,NaN,NaN,NaN
2,AFG,NaN,NaN,NaN,NaN
3,AFW,8.525270e+11,NaN,NaN,NaN
4,AGO,1.221749e+11,NaN,1084.0,NaN


## 9 · Event Discovery

Three-gate pipeline that surfaces oil-relevant events from the news **without pre-specifying
geography or event type**. The data tells you what happened and where — you don't tell it
what to look for.

| Gate | Question | Method |
|---|---|---|
| **1 — News spike** | Is something being talked about significantly more? | Country mentions in GDELT titles: last 30d vs prior 60d, ≥5× ratio |
| **2 — Oil relevance** | Does this geography matter for oil supply? | JODI producer list (≥100 kbd) + chokepoint-adjacent countries |
| **3 — Market signal** | Did the market notice? | Brent/WTI or PortWatch tanker transit move >2% in same window |

**Output:** `INVESTIGATE` (all 3 gates) or `WATCH` (Gates 1–2, no market move yet).
The event's geography and direction (price up/down, transit up/down) come from the data.


In [71]:
import re

# ── Build name → ISO-3 lookup from pycountry ────────────────────────────────
_COUNTRY_NAME_MAP = {}
for _c in pycountry.countries:
    _COUNTRY_NAME_MAP[_c.name.lower()] = _c.alpha_3
    if hasattr(_c, "common_name"):
        _COUNTRY_NAME_MAP[_c.common_name.lower()] = _c.alpha_3
    if hasattr(_c, "official_name"):
        _COUNTRY_NAME_MAP[_c.official_name.lower()] = _c.alpha_3

# Aliases: short forms, adjectives, actor/group names, and chokepoint terms.
# Value = ISO-3 code, or None for multi-country concepts (skip those).
COUNTRY_ALIASES = {
    # English short forms & adjectives
    "us": "USA",  "u.s.": "USA",  "usa": "USA",  "american": "USA",
    "uk": "GBR",  "u.k.": "GBR",  "britain": "GBR",   "british": "GBR",
    "uae": "ARE", "emirati": "ARE", "emirates": "ARE",
    "iranian": "IRN",  "iraqi": "IRQ",   "saudi": "SAU",
    "russian": "RUS",  "chinese": "CHN", "venezuelan": "VEN",
    "nigerian": "NGA", "libyan": "LBY",  "kuwaiti": "KWT",
    "qatari": "QAT",   "omani": "OMN",   "bahraini": "BHR",
    "yemeni": "YEM",   "sudanese": "SDN","azerbaijani": "AZE",
    "kazakh": "KAZ",   "algerian": "DZA","angolan": "AGO",
    "egyptian": "EGY", "turkish": "TUR", "lebanese": "LBN",
    "syrian": "SYR",   "ukrainian": "UKR","indonesian": "IDN",
    "malaysian": "MYS","panamanian": "PAN","norwegian": "NOR",
    # Actor / group → country they operate from
    "houthi":    "YEM", "houthis":   "YEM",
    "hezbollah": "LBN",
    "hamas":     "PSE",
    "opec":      None,   # multi-country — skip
    # Chokepoint terms → the country that controls/sits on it
    "suez":          "EGY",
    "malacca":       "MYS",
    "hormuz":        "IRN",
    "bab el-mandeb": "YEM", "bab-el-mandeb": "YEM",
    "panama canal":  "PAN",
}


def extract_countries_from_title(title):
    """
    Return a list of ISO-3 codes for countries/regions mentioned in an article title.
    Checks COUNTRY_ALIASES first (longer / more specific phrases), then pycountry names.
    Skips names shorter than 4 chars to avoid false matches on common words.
    """
    t = str(title).lower()
    found = set()

    for alias, iso3 in COUNTRY_ALIASES.items():
        if iso3 and re.search(r"\b" + re.escape(alias) + r"\b", t):
            found.add(iso3)

    for name, iso3 in _COUNTRY_NAME_MAP.items():
        if len(name) >= 4 and re.search(r"\b" + re.escape(name) + r"\b", t):
            found.add(iso3)

    return list(found)


In [72]:
def detect_spikes(df, primary_days=30, threshold=5.0, min_recent=3):
    """
    Gate 1: find countries mentioned ≥ threshold× more often in the last primary_days
    than in the prior context window (normalised to the same length).

    min_recent guards against tiny-count noise (e.g. 0 → 1 article counting as infinite spike).
    Returns a DataFrame: iso3, recent_count, prior_count, prior_norm, spike_ratio.
    """
    if df.empty or "seendate" not in df.columns or "title" not in df.columns:
        return pd.DataFrame()

    df = df.copy()
    df["countries"] = df["title"].apply(extract_countries_from_title)

    # one row per country mention per article
    exploded = (df.explode("countries")
                  .dropna(subset=["countries"])
                  .rename(columns={"countries": "iso3"}))

    anchor = exploded["seendate"].max()
    cutoff = anchor - pd.Timedelta(days=primary_days)
    start  = exploded["seendate"].min()

    recent = exploded[exploded["seendate"] >= cutoff]
    prior  = exploded[exploded["seendate"] <  cutoff]

    recent_counts = recent.groupby("iso3").size().rename("recent_count")
    prior_counts  = prior.groupby("iso3").size().rename("prior_count")

    combined = pd.concat([recent_counts, prior_counts], axis=1).fillna(0)

    # normalise prior window to the same length as the primary window
    prior_days = max((cutoff - start).days, 1)
    combined["prior_norm"] = (combined["prior_count"] / prior_days) * primary_days

    # +0.5 smoothing: dampens 0→small jumps and prevents division-by-zero
    combined["spike_ratio"] = combined["recent_count"] / (combined["prior_norm"] + 0.5)

    result = (combined
              .loc[(combined["spike_ratio"] >= threshold) &
                   (combined["recent_count"] >= min_recent)]
              .sort_values("spike_ratio", ascending=False)
              .reset_index())

    print(f"  Gate 1: {len(result)} countries spiked ≥{threshold}× "
          f"(last {primary_days}d vs prior {prior_days}d, min {min_recent} recent articles)")
    return result


In [73]:
# Countries that sit on or adjacent to a major chokepoint.
# An event here is oil-relevant even if the country isn't a top producer.
CHOKEPOINT_ADJACENT = {
    "YEM": "Bab el-Mandeb", "DJI": "Bab el-Mandeb", "ERI": "Bab el-Mandeb",
    "EGY": "Suez Canal",    "ISR": "Suez Canal",
    "IRN": "Strait of Hormuz", "OMN": "Strait of Hormuz",
    "MYS": "Strait of Malacca", "SGP": "Strait of Malacca", "IDN": "Strait of Malacca",
    "PAN": "Panama Canal",
    "TUR": "Turkish Straits",
    "DNK": "Danish Straits",  "SWE": "Danish Straits",
}


def filter_oil_relevant(spikes_df, jodi_df, min_prod_kbd=100):
    """
    Gate 2: keep only spiking countries that are either meaningful crude producers
    (JODI latest value ≥ min_prod_kbd thousand barrels/day) or sit on a chokepoint.

    Adds 'relevance' and 'chokepoint' columns.
    """
    if spikes_df.empty:
        return pd.DataFrame()

    # derive producer set from JODI
    producers = set()
    if (not jodi_df.empty
            and "iso3" in jodi_df.columns
            and "OBS_VALUE" in jodi_df.columns):
        latest_prod = (jodi_df.sort_values("TIME_PERIOD")
                               .groupby("iso3")["OBS_VALUE"].last())
        producers = set(latest_prod[latest_prod >= min_prod_kbd].index)

    adjacent = set(CHOKEPOINT_ADJACENT)
    result = spikes_df[spikes_df["iso3"].isin(producers | adjacent)].copy()

    result["relevance"] = result["iso3"].apply(
        lambda x: ("producer+chokepoint" if x in producers and x in adjacent
                   else "producer"         if x in producers
                   else "chokepoint-adjacent")
    )
    result["chokepoint"] = result["iso3"].map(CHOKEPOINT_ADJACENT)

    print(f"  Gate 2: {len(result)} oil-relevant candidates "
          f"({len(producers)} known producers, {len(adjacent)} chokepoint-adjacent countries in scope)")
    return result.reset_index(drop=True)


In [74]:
def check_comovement(eia_df, portwatch_df, primary_days=30):
    """
    Gate 3: did Brent, WTI, or chokepoint tanker transits move over the primary window?

    Compares the mean value in the last primary_days to the equivalent prior window.
    A move of >2% in either direction counts as a market signal.

    Returns (signals dict, any_signal bool).
    """
    signals = {}

    # ── Price signals from EIA bulk ──────────────────────────
    for metric in ["brent_usd_bbl", "wti_usd_bbl"]:
        if eia_df.empty:
            break
        sub = (eia_df[eia_df["metric"] == metric]
               .sort_values("date")
               .dropna(subset=["value"]))
        if sub.empty:
            continue
        anchor = sub["date"].max()
        cutoff = anchor - pd.Timedelta(days=primary_days)
        recent = sub[sub["date"] >= cutoff]["value"]
        prior  = sub[sub["date"] <  cutoff].tail(primary_days)["value"]
        if recent.empty or prior.empty:
            continue
        move = recent.mean() - prior.mean()
        pct  = move / prior.mean() * 100 if prior.mean() else 0
        signals[metric] = {
            "direction": "up" if move > 0 else "down",
            "move":      round(move, 2),
            "move_pct":  round(pct, 1),
            "latest":    round(sub["value"].iloc[-1], 2),
        }

    # ── Transit signals from PortWatch ──────────────────────
    if not portwatch_df.empty and "portname" in portwatch_df.columns:
        for cp in portwatch_df["portname"].unique():
            sub = (portwatch_df[portwatch_df["portname"] == cp]
                   .sort_values("date")
                   .dropna(subset=["n_tanker"]))
            if sub.empty:
                continue
            anchor = sub["date"].max()
            cutoff = anchor - pd.Timedelta(days=primary_days)
            recent = sub[sub["date"] >= cutoff]["n_tanker"]
            prior  = sub[sub["date"] <  cutoff].tail(primary_days)["n_tanker"]
            if recent.empty or prior.empty:
                continue
            move = recent.mean() - prior.mean()
            pct  = move / prior.mean() * 100 if prior.mean() else 0
            signals[f"tanker_{cp}"] = {
                "direction": "up" if move > 0 else "down",
                "move":      round(move, 1),
                "move_pct":  round(pct, 1),
                "latest":    round(recent.mean(), 1),
            }

    # >2% move in either direction = market noticed something
    any_signal = any(abs(v["move_pct"]) > 2 for v in signals.values())

    print(f"  {'Signal detected' if any_signal else 'No significant market movement'} "
          f"(>{primary_days}d window, threshold >2%)")
    for name, v in signals.items():
        print(f"    {name}: {v['direction'].upper()} {abs(v['move_pct']):.1f}%  "
              f"(latest: {v['latest']})")

    return signals, any_signal


In [75]:
def discover_events(primary_days=30, context_days=90, spike_threshold=5.0):
    """
    Event discovery pipeline — three gates → ranked candidate events.

    INVESTIGATE = passed all 3 gates (news spike + oil-relevant geography + market moved).
    WATCH       = passed Gates 1 & 2 but no market signal yet.

    Returns a DataFrame of candidates.
    """
    print("=" * 60)
    print("EVENT DISCOVERY")
    print(f"Primary window : {primary_days}d  |  Context : {context_days}d  |  Spike threshold : {spike_threshold}×")
    print("=" * 60)

    # ── Gate 1 ──────────────────────────────────────────────
    print("\nGate 1 — News volume spike...")
    if gdelt_discovery.empty:
        print("  GDELT data unavailable. Re-run the GDELT cell or check connection.")
        return pd.DataFrame()
    spikes = detect_spikes(gdelt_discovery, primary_days=primary_days, threshold=spike_threshold)

    if spikes.empty:
        print("  No spikes found. Nothing to investigate.")
        return pd.DataFrame()

    # ── Gate 2 ──────────────────────────────────────────────
    print("\nGate 2 — Oil-relevance filter...")
    candidates = filter_oil_relevant(spikes, jodi)

    if candidates.empty:
        print("  No oil-relevant events — spiking countries are neither producers nor chokepoint-adjacent.")
        return pd.DataFrame()

    # ── Gate 3 ──────────────────────────────────────────────
    print("\nGate 3 — Market co-movement...")
    market_signals, any_signal = check_comovement(eia, portwatch, primary_days=primary_days)

    candidates["classification"] = "WATCH"
    if any_signal:
        candidates["classification"] = "INVESTIGATE"

    # attach Brent summary to each candidate row for quick scanning
    brent = market_signals.get("brent_usd_bbl", {})
    candidates["brent_move_pct"]  = brent.get("move_pct")
    candidates["brent_direction"] = brent.get("direction")

    # ── Output ───────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("CANDIDATE EVENTS")
    print("=" * 60)
    show = ["iso3", "relevance", "chokepoint", "recent_count",
            "spike_ratio", "brent_move_pct", "brent_direction", "classification"]
    show = [c for c in show if c in candidates.columns]
    print(candidates[show].to_string(index=False))
    print()
    return candidates

candidates = discover_events()


EVENT DISCOVERY
Primary window : 30d  |  Context : 90d  |  Spike threshold : 5.0×

Gate 1 — News volume spike...
  Gate 1: 8 countries spiked ≥5.0× (last 30d vs prior 1d, min 3 recent articles)

Gate 2 — Oil-relevance filter...
  Gate 2: 8 oil-relevant candidates (35 known producers, 14 chokepoint-adjacent countries in scope)

Gate 3 — Market co-movement...
  Signal detected (>30d window, threshold >2%)
    brent_usd_bbl: DOWN 28.2%  (latest: 81.62)
    wti_usd_bbl: DOWN 25.2%  (latest: 79.2)
    tanker_Suez Canal: UP 9.2%  (latest: 16.5)
    tanker_Panama Canal: UP 12.3%  (latest: 15.4)
    tanker_Bab el-Mandeb Strait: DOWN 8.0%  (latest: 13.2)
    tanker_Malacca Strait: UP 5.7%  (latest: 72.9)
    tanker_Strait of Hormuz: UP 457.9%  (latest: 9.5)

CANDIDATE EVENTS
iso3           relevance       chokepoint  recent_count  spike_ratio  brent_move_pct brent_direction classification
 IRN producer+chokepoint Strait of Hormuz            87        174.0           -28.2            down    INV

## 8 · War-game trace — five-layer cascade

`run_top_candidate()` picks the top-ranked event from §9's discovery pipeline and runs
`trace_event()` on it automatically. **No chokepoint is hardcoded** — the geography comes
from what the data surfaced.

The five layers, in the order the method demands:

1. **① DETECT** — GDELT news volume. The event tells you *where* to look; it does not prove supply moved.
2. **② VERIFY GATE** — IMF PortWatch tanker transits + EIA crude stocks. Separates a **real physical disruption** (transits fell) from a **risk premium** (fear/price, no physical cut). Only a confirmed drop justifies reading the rest as live.
3. **③ SUPPLY & PRICE** — crude at risk (JODI), Brent/WTI move (EIA), geopolitical risk (GPR).
4. **④ JET FUEL** — US Gulf Coast jet-fuel spot (EIA); crack spread over Brent.
5. **⑤ DOWNSTREAM** — exposed crude importers (Comtrade) → airfields in those countries (OurAirports).


In [76]:
# Which producing countries mainly ship through each chokepoint (ISO-3).
# Editable — drawn from EIA chokepoint analysis.
CHOKEPOINT_EXPORTERS = {
    "Hormuz":      ["SAU","IRN","IRQ","KWT","ARE","QAT","BHR"],
    "Malacca":     ["SAU","ARE","KWT","IRQ"],
    "Suez":        ["SAU","IRQ","RUS","KAZ"],
    "BabelMandeb": ["SAU","IRQ","RUS","KAZ"],
    "Panama":      ["USA","COL","ECU"],
}

# Reverse map: PortWatch full name -> trace_event() short name
PW_TO_SHORT = {v: k for k, v in PW_NAME.items()}


# -- small helpers -------------------------------------------------------------
def _series(metric):
    if "eia" not in globals() or eia.empty:
        return pd.DataFrame(columns=["date", "value"])
    return eia[eia["metric"] == metric].sort_values("date")

def _move(metric, n):
    s = _series(metric)
    if len(s) <= n:
        return (s["value"].iloc[-1] if len(s) else None, None)
    now, then = s["value"].iloc[-1], s["value"].iloc[-1 - n]
    return now, (now / then - 1) * 100 if then else None

def _news_spike(news, recent_days=3, window_days=21):
    if news is None or news.empty or "seendate" not in news:
        return None
    d = news.dropna(subset=["seendate"]).copy()
    if d.empty:
        return None
    d["day"] = d["seendate"].dt.floor("D")
    end     = d["day"].max()
    span    = d[d["day"] > end - pd.Timedelta(days=window_days)]
    per_day = span.groupby("day").size()
    recent  = per_day[per_day.index > end - pd.Timedelta(days=recent_days)].mean()
    base    = per_day[per_day.index <= end - pd.Timedelta(days=recent_days)].mean()
    recent  = 0 if pd.isna(recent) else recent
    base    = 0 if pd.isna(base)   else base
    ratio   = (recent / base) if base else (float("inf") if recent else 0)
    return {"recent": recent, "base": base, "ratio": ratio, "spike": ratio >= 1.5}

def _gate(pw_name, recent_days=7, drop_last=1, pre_lo=14, pre_hi=45, thresh=0.7):
    if not pw_name or "portwatch" not in globals() or portwatch.empty:
        return None
    s = (portwatch[portwatch["portname"] == pw_name]
         .sort_values("date").set_index("date")["n_tanker"])
    if len(s) < pre_hi + recent_days:
        return {"insufficient": True, "n": len(s)}
    end    = s.index.max()
    r_end  = end - pd.Timedelta(days=drop_last)
    recent = s.loc[r_end - pd.Timedelta(days=recent_days) : r_end].mean()
    pre    = s.loc[r_end - pd.Timedelta(days=pre_hi)      : r_end - pd.Timedelta(days=pre_lo)].mean()
    ratio  = (recent / pre) if pre else float("nan")
    return {"pw_name": pw_name, "recent": recent, "pre": pre, "ratio": ratio,
            "disrupted": (ratio < thresh)}

def _country_label(iso3):
    """Return 'Full Name (ISO3)' using pycountry, or just ISO3 if lookup fails."""
    try:
        name = pycountry.countries.get(alpha_3=iso3).name
        if len(name) > 20:
            name = name[:18] + ".."
        return "{} ({})".format(name, iso3)
    except Exception:
        return iso3


def trace_event(chokepoint, date, days_of_cover=None):
    exporters = CHOKEPOINT_EXPORTERS.get(chokepoint, [])
    pw_name   = PW_NAME.get(chokepoint)
    head      = []
    print("=" * 80)
    print("WAR-GAME TRACE  .  {}  .  as of {}".format(chokepoint, date))
    print("=" * 80)

    # -- (1) News signal -------------------------------------------------------
    print("\n(1) NEWS SIGNAL   (how much is this event dominating media coverage?)")
    sig = _news_spike(gdelt_discovery) if "gdelt_discovery" in globals() else None
    if sig:
        if sig["base"] == 0:
            print("   Recent articles per day : {:.0f}".format(sig["recent"]))
            print("   Historical baseline     : none available")
            print("   Why: the API returned only recent articles -- older ones were cut off by the")
            print("   250-article limit. Without older articles there is nothing to compare against,")
            print("   so no meaningful ratio can be computed.")
            print("   Signal direction: coverage is ELEVATED but the spike ratio is unreliable.")
            head.append("news coverage ELEVATED (no historical baseline available)")
        else:
            flag = "ELEVATED -- abnormally high coverage" if sig["spike"] else "normal coverage level"
            r    = "{:.1f}x".format(sig["ratio"])
            print("   Recent articles per day : {:.0f}".format(sig["recent"]))
            print("   Historical baseline/day : {:.0f}".format(sig["base"]))
            print("   Ratio                   : {}  ({})".format(r, flag))
            head.append("news {}".format(flag.split("--")[0].strip()))
    else:
        print("   No news data available.")
    print("   Note: a news spike shows WHERE to look -- it does not confirm supply was disrupted.")

    # -- (2) Physical disruption check -----------------------------------------
    print("\n(2) PHYSICAL DISRUPTION CHECK   (did ships actually stop moving?)")
    print("   Source: IMF PortWatch -- counts tanker ships passing through the chokepoint each day.")
    g = _gate(pw_name)
    if g is None:
        print("   No PortWatch data available for this chokepoint.")
    elif g.get("insufficient"):
        print("   Only {} days of transit data -- not enough history to judge.".format(g["n"]))
    else:
        pct = g["ratio"] * 100
        print("   Chokepoint: {}".format(g["pw_name"]))
        print("   Tanker ships in the last 7 days   : {:.1f} per day".format(g["recent"]))
        print("   Tanker ships before the event     : {:.1f} per day  "
              "(average from 14 to 45 days ago)".format(g["pre"]))
        print("   Current traffic as % of normal    : {:.0f}%".format(pct))
        if g["disrupted"]:
            verdict = "CONFIRMED PHYSICAL DISRUPTION -- tanker traffic has collapsed"
            head.append("tanker traffic at {:.0f}% of normal -- physical disruption confirmed".format(pct))
        else:
            verdict = "NO PHYSICAL DISRUPTION -- traffic is normal; any price move is fear-driven"
            head.append("tanker traffic normal -- price move is risk premium, not a real supply cut")
        print("   Result: {}".format(verdict))

    st = _series("us_crude_stocks_kbbl").tail(8)
    if len(st) >= 2:
        chg   = st["value"].iloc[-1] - st["value"].iloc[0]
        trend = "falling" if chg < 0 else "rising"
        print()
        print("   US crude oil in storage: {:,.0f} thousand barrels  "
              "({} by {:+,.0f} over {} weeks)".format(
              st["value"].iloc[-1], trend, chg, len(st) - 1))
        if chg < 0:
            print("   Falling stockpiles indicate that supply is not keeping up with demand.")

    if g and g.get("disrupted") is True:
        print()
        print("   >>> Disruption confirmed -- treat the supply and price numbers below as a LIVE event.")
    elif g and g.get("disrupted") is False:
        print()
        print("   >>> No confirmed disruption -- the analysis below is a WHAT-IF, not a live event.")

    # -- (3) Supply and price --------------------------------------------------
    print("\n(3) CRUDE OIL SUPPLY & PRICE")
    _brent_s = _series("brent_usd_bbl")
    if not _brent_s.empty:
        _eia_date = _brent_s["date"].max()
        _age_d    = int((pd.Timestamp.today() - _eia_date).days)
        _stale    = "  WARNING: DATA IS STALE -- prices shown are not current" if _age_d > 5 else ""
        print("   [Price data last updated: {}  ({} days ago){}]".format(
              _eia_date.date(), _age_d, _stale))

    if "crude_prod_latest" in country.columns:
        at_risk = country[country["iso3"].isin(exporters)]["crude_prod_latest"].sum()
        world   = country["crude_prod_latest"].sum()
        share   = " ({:.0f}% of total world output)".format(at_risk/world*100) if world else ""
        print()
        print("   Daily crude oil output from affected exporters: {:,.0f} thousand barrels/day{}".format(
              at_risk, share))
        print("   This is the volume that normally ships through the disrupted chokepoint.")
        print("   Note: production data has a ~2 month publication lag -- actual output may differ.")
        head.append("{:,.0f} thousand barrels/day of supply at risk".format(at_risk))

    print()
    # Brent = the main international oil price benchmark (crude from the North Sea).
    # WTI   = the US domestic benchmark (West Texas Intermediate); usually $2-4 below Brent.
    for m, lab in [("brent_usd_bbl", "Brent (global oil price benchmark)"),
                   ("wti_usd_bbl",   "WTI   (US domestic oil price benchmark)")]:
        now, wk = _move(m, 5)
        if now is not None:
            mv = "  ({:+.1f}% vs one week ago)".format(wk) if wk is not None else ""
            print("   {}: ${:,.2f} per barrel{}".format(lab, now, mv))
            if "Brent" in lab and wk is not None:
                head.append("Brent oil price {:+.1f}% over the past week".format(wk))

    if "gpr" in globals() and not gpr.empty:
        cur = gpr["GPR"].iloc[-1]
        pct = (gpr["GPR"] < cur).mean() * 100
        print()
        print("   Geopolitical Risk Index: {:.0f}  "
              "(higher than {:.0f}% of all months ever recorded)".format(cur, pct))
        print("   Measures how much global conflict and political tension is in the news.")

    # -- (4) Jet fuel ----------------------------------------------------------
    print("\n(4) JET FUEL PRICE")
    jet, jwk  = _move("jet_fuel_usd_gal", 5)
    brent_ser = _series("brent_usd_bbl")
    if jet is not None:
        jet_bbl = jet * 42   # 1 barrel = 42 US gallons
        print("   US Gulf Coast jet fuel: ${:,.3f} per gallon  "
              "(${:,.2f} per barrel -- 1 barrel = 42 gallons)".format(jet, jet_bbl))
        if jwk is not None:
            print("   Change vs one week ago: {:+.1f}%".format(jwk))
        if len(brent_ser):
            crack = jet_bbl - brent_ser["value"].iloc[-1]
            print("   Premium over raw crude oil: ${:,.2f} per barrel".format(crack))
            print("   (Jet fuel costs more than crude because it must be refined.")
            print("    A rising premium means refining capacity or distribution is under strain.)")
    else:
        print("   Jet fuel price data not available.")

    # -- (5) Countries and airports at risk ------------------------------------
    print("\n(5) COUNTRIES AND AIRPORTS AT RISK")
    print()
    print("   How this section works:")
    print("   1. Find which countries import crude oil from the affected exporters")
    print("      (using UN Comtrade bilateral trade data)")
    print("   2. Score each country on how exposed it is:")
    print("      - What fraction of its crude imports come from the disrupted exporters?")
    print("      - Does it produce enough oil domestically to offset a supply cut?")
    print("   3. Count the airports in each country -- those airports face potential fuel supply risk")
    print()
    print("   Risk tiers:")
    print("   CRITICAL -- imports most of its crude from the disrupted exporters")
    print("               AND produces little oil domestically (no fallback supply)")
    print("   ELEVATED -- moderate exposure to disrupted supply")
    print("               OR limited domestic production to buffer a supply cut")
    print("   WATCH    -- some exposure but domestic production or low share provides cover")

    bilateral_exp = pd.DataFrame()
    if (not crude_imports.empty
            and "partner_iso3" in crude_imports.columns
            and "iso3" in crude_imports.columns):
        from_exp = (crude_imports[crude_imports["partner_iso3"].isin(exporters)]
                    .groupby("iso3")["primaryValue"].sum()
                    .rename("import_from_exporters"))
        total_imp = (crude_imports.groupby("iso3")["primaryValue"].sum()
                     .rename("import_total"))
        exp_df = pd.concat([from_exp, total_imp], axis=1).fillna(0)
        exp_df["import_dep_share"] = (
            exp_df["import_from_exporters"]
            / exp_df["import_total"].replace(0, float("nan"))
        )
        bilateral_exp = exp_df[exp_df["import_from_exporters"] > 0].reset_index()
        if "refYear" in crude_imports.columns and "refMonth" in crude_imports.columns:
            _ct_yr = crude_imports["refYear"].iloc[0]
            _ct_mo = crude_imports["refMonth"].iloc[0]
            print()
            print("   Trade data: {}-{:02d}  (Comtrade free tier, capped at 2,500 rows)".format(
                  int(_ct_yr), int(_ct_mo)))
            print("   WARNING: Large importers like China, India, and South Korea may be missing")
            print("   because the free API cap cuts off before reaching all bilateral records.")

    if bilateral_exp.empty and "crude_import_value" in country.columns:
        fallback = (country.dropna(subset=["crude_import_value"])
                           .sort_values("crude_import_value", ascending=False).head(15))
        bilateral_exp = fallback[["iso3", "crude_import_value"]].copy().rename(
            columns={"crude_import_value": "import_from_exporters"}
        )
        bilateral_exp["import_total"]     = bilateral_exp["import_from_exporters"]
        bilateral_exp["import_dep_share"] = float("nan")
        print("   (Using total import value as fallback -- no bilateral source breakdown available)")

    if bilateral_exp.empty:
        print("   No trade data available to identify exposed importers.")
    else:
        ctx_cols = [c for c in ["crude_prod_latest", "oil_rents_pct_gdp", "gdp_usd"]
                    if c in country.columns]
        exp = bilateral_exp.merge(country[["iso3"] + ctx_cols], on="iso3", how="left")
        exp["crude_prod_latest"] = exp["crude_prod_latest"].fillna(0)

        TIER_RANK = {"CRITICAL": 0, "ELEVATED": 1, "WATCH": 2}

        def _classify_row(dep, prod):
            if pd.isna(dep):
                return "WATCH" if prod >= 500 else "ELEVATED"
            if dep >= 0.25 and prod < 100:
                return "CRITICAL"
            if dep >= 0.10 or (dep >= 0.05 and prod < 500):
                return "ELEVATED"
            return "WATCH"

        exp["tier"] = [
            _classify_row(r["import_dep_share"], r["crude_prod_latest"])
            for _, r in exp.iterrows()
        ]
        exp["tier_rank"] = exp["tier"].map(TIER_RANK)

        ap_counts = (airports.groupby("iso3")
                             .agg(total_airports=("ident", "count"),
                                  large_airports=("type", lambda x: (x == "large_airport").sum()))
                             .reset_index())
        exp = exp.merge(ap_counts, on="iso3", how="left")
        exp["large_airports"] = exp["large_airports"].fillna(0).astype(int)
        exp["total_airports"] = exp["total_airports"].fillna(0).astype(int)
        exp = (exp.sort_values(["tier_rank", "import_from_exporters"], ascending=[True, False])
                  .reset_index(drop=True)
                  .drop(columns=["tier_rank"]))

        us_doc = None
        if "eia" in globals() and not eia.empty:
            stk = eia[eia["metric"] == "us_crude_stocks_kbbl"].sort_values("date")
            imp = eia[eia["metric"] == "us_crude_imports_kbd"].sort_values("date")
            if not stk.empty and not imp.empty and imp["value"].iloc[-1] > 0:
                us_doc = stk["value"].iloc[-1] / imp["value"].iloc[-1]

        W0, W1, W2, W3, W4, W5 = 26, 22, 24, 24, 14, 12

        print()
        print("   Column guide:")
        print("     'Crude Bought from Affected Exporters ($M)'")
        print("       Dollar value of crude oil imported from the exporters that ship through")
        print("       the disrupted chokepoint, in the most recent Comtrade period.")
        print("     '% of This Country\'s Total Crude Imports'")
        print("       How large a share of their total crude purchases that represents.")
        print("       Higher percentage = more dependent on the disrupted supply.")
        print("     'Own Crude Production (thousand barrels/day)'")
        print("       How much crude oil this country produces itself.")
        print("       A country that produces a lot can offset an import cut.")
        print("     'Large Airports'")
        print("       Airports capable of handling commercial jet aircraft (OurAirports data).")

        hdr  = ("  {:<{w0}}  {:>{w1}}  {:>{w2}}  {:>{w3}}  {:>{w4}}  {:>{w5}}".format(
                "Country", "Crude Bought from", "% of This Country's", "Own Crude Production",
                "Large", "All", w0=W0, w1=W1, w2=W2, w3=W3, w4=W4, w5=W5))
        hdr2 = ("  {:<{w0}}  {:>{w1}}  {:>{w2}}  {:>{w3}}  {:>{w4}}  {:>{w5}}".format(
                "", "Affected Exp. ($M)", "Total Crude Imports", "(thousand bbl/day)",
                "Airports", "Airports", w0=W0, w1=W1, w2=W2, w3=W3, w4=W4, w5=W5))
        sep  = "  " + "  ".join(["-"*W0, "-"*W1, "-"*W2, "-"*W3, "-"*W4, "-"*W5])

        for tier in ["CRITICAL", "ELEVATED", "WATCH"]:
            rows = exp[exp["tier"] == tier]
            if rows.empty:
                continue
            if tier == "CRITICAL":
                desc = "heavily dependent on disrupted supply + almost no domestic production"
            elif tier == "ELEVATED":
                desc = "meaningful exposure to disrupted supply OR limited domestic buffer"
            else:
                desc = "some exposure but domestic production or low share provides cover"
            print()
            print("  [{}]  {}".format(tier, desc))
            print(hdr)
            print(hdr2)
            print(sep)
            for _, r in rows.iterrows():
                label  = _country_label(r["iso3"])
                dep_s  = ("{:.0f}%".format(r["import_dep_share"] * 100)
                          if pd.notna(r["import_dep_share"]) else "unknown")
                prod_s = ("{:,.0f}".format(r["crude_prod_latest"])
                          if r["crude_prod_latest"] > 0 else "near zero")
                note   = ""
                if r["iso3"] == "USA" and us_doc is not None:
                    note = "  ({:.0f} days of crude in storage)".format(us_doc)
                print("  {:<{w0}}  {:>{w1},.1f}  {:>{w2}}  {:>{w3}}  {:>{w4}}  {:>{w5}}{}".format(
                      label,
                      r["import_from_exporters"] / 1e6,
                      dep_s, prod_s,
                      r["large_airports"], r["total_airports"], note,
                      w0=W0, w1=W1, w2=W2, w3=W3, w4=W4, w5=W5))

        n_crit   = int((exp["tier"] == "CRITICAL").sum())
        n_elev   = int((exp["tier"] == "ELEVATED").sum())
        n_wtch   = int((exp["tier"] == "WATCH").sum())
        lrg_risk = int(exp[exp["tier"].isin(["CRITICAL", "ELEVATED"])]["large_airports"].sum())
        print()
        print("  Summary:")
        print("    Countries at CRITICAL risk : {}".format(n_crit))
        print("    Countries at ELEVATED risk : {}".format(n_elev))
        print("    Countries at WATCH         : {}".format(n_wtch))
        print("    Large airports in CRITICAL + ELEVATED countries: {:,}".format(lrg_risk))
        if us_doc is not None:
            print("    US crude oil stockpile covers approx. {:.0f} days at current import rates".format(
                  us_doc))
        if days_of_cover:
            print("    Analyst-provided days of fuel cover: {}".format(days_of_cover))

        exposed_iso3 = set(exp["iso3"])
        head.append("{} countries CRITICAL / {} ELEVATED".format(n_crit, n_elev))

    print("\n" + "-" * 80)
    print("OVERALL ASSESSMENT:")
    for item in head:
        print("  .  {}".format(item))
    if not head:
        print("  Insufficient data to assess.")
    print("=" * 80)


def run_top_candidate(candidates_df, date=None, days_of_cover=None):
    """Run trace_event() on the top candidate from discover_events()."""
    import datetime
    if candidates_df is None or candidates_df.empty:
        print("No candidates found. Run the Event Discovery section first.")
        return
    date     = date or datetime.date.today().isoformat()
    inv      = candidates_df[candidates_df["classification"] == "INVESTIGATE"]
    top      = inv.iloc[0] if not inv.empty else candidates_df.iloc[0]
    iso3     = top["iso3"]
    full_cp  = top.get("chokepoint") if pd.notna(top.get("chokepoint")) else None
    short_cp = PW_TO_SHORT.get(full_cp) if full_cp else None
    if not short_cp:
        for cp, exporters in CHOKEPOINT_EXPORTERS.items():
            if iso3 in exporters:
                short_cp = cp
                break
    if not short_cp:
        print("Top candidate {} does not map to a known chokepoint.".format(iso3))
        print("Add it to CHOKEPOINT_EXPORTERS to enable a full trace.")
        return
    label = _country_label(iso3)
    print("Top event candidate : {}  (news spike {:.0f}x above baseline)".format(
          label, top["spike_ratio"]))
    print("Chokepoint          : {}".format(full_cp or short_cp))
    print("Classification      : {}".format(top["classification"]))
    print()
    trace_event(short_cp, date, days_of_cover=days_of_cover)


In [77]:
import datetime as _dt

# ═══════════════════════════════════════════════════════════════════════════════
# CONTROL PANEL  —  edit the values below, then re-run this cell
# ═══════════════════════════════════════════════════════════════════════════════

# ── What to trace ─────────────────────────────────────────────────────────────
# "auto"         → top INVESTIGATE candidate by spike ratio  [DEFAULT]
# "all"          → every INVESTIGATE candidate, one trace each
# "<ISO-3 code>" → a specific country (must appear in the candidate table)
TRACE_TARGET = "auto"

# ── Days of cover (buffer before shortage bites) ──────────────────────────────
# Set to a number if known (source: OPEC MOMR or EIA). None = omit from output.
DAYS_OF_COVER = None

# ── Discovery settings (re-run §9 after changing these) ───────────────────────
# PRIMARY_DAYS    = 30    # news comparison window (days)   [currently: 30]
# SPIKE_THRESHOLD = 5.0   # minimum spike ratio to flag     [currently: 5.0×]

# ═══════════════════════════════════════════════════════════════════════════════

if "candidates" not in dir() or candidates.empty:
    print("No candidates found. Run §9 (discover_events) first.")
else:
    # ── Candidate summary ─────────────────────────────────────────────────────
    print("=" * 72)
    print("DISCOVERED CANDIDATES")
    print("  {:<6}  {:<24}  {:<22}  {:>7}  {:>10}  {}".format(
          "ISO3", "Relevance", "Chokepoint", "Spike", "Brent 30d", "Class"))
    print("  {}  {}  {}  {}  {}  {}".format(
          "-"*6, "-"*24, "-"*22, "-"*7, "-"*10, "-"*12))
    for _, row in candidates.iterrows():
        cp    = str(row["chokepoint"]) if pd.notna(row.get("chokepoint")) else "-"
        brent = ("{:+.1f}%".format(row["brent_move_pct"])
                 if pd.notna(row.get("brent_move_pct")) else "-")
        print("  {:<6}  {:<24}  {:<22}  {:>6.0f}x  {:>10}  {}".format(
              row["iso3"], row["relevance"], cp,
              row["spike_ratio"], brent, row["classification"]))
    print()

    # dynamically show available options from actual candidates
    inv_codes = list(candidates[candidates["classification"] == "INVESTIGATE"]["iso3"])
    wat_codes = list(candidates[candidates["classification"] == "WATCH"]["iso3"])
    inv_str   = ", ".join('"{}"'.format(c) for c in inv_codes)
    wat_str   = ", ".join('"{}"'.format(c) for c in wat_codes)

    print("  To trace a specific candidate, change TRACE_TARGET above to one of:")
    print("    INVESTIGATE : " + inv_str)
    if wat_codes:
        print("    WATCH       : " + wat_str)
    print('    All at once : "all"')
    print()
    print("  Current settings:")
    print('    TRACE_TARGET  = "{}"'.format(TRACE_TARGET))
    print("    DAYS_OF_COVER = {}".format(DAYS_OF_COVER))
    print("=" * 72)
    print()

    # ── Run ───────────────────────────────────────────────────────────────────
    if TRACE_TARGET == "auto":
        run_top_candidate(candidates, days_of_cover=DAYS_OF_COVER)

    elif TRACE_TARGET == "all":
        inv     = candidates[candidates["classification"] == "INVESTIGATE"]
        targets = inv if not inv.empty else candidates
        for _, row in targets.iterrows():
            run_top_candidate(
                candidates[candidates["iso3"] == row["iso3"]],
                days_of_cover=DAYS_OF_COVER,
            )
            print()

    else:
        subset = candidates[candidates["iso3"] == TRACE_TARGET.upper()]
        if subset.empty:
            print('  "{}" not found in candidates.'.format(TRACE_TARGET))
            print("  Available ISO-3 codes: {}".format(list(candidates["iso3"])))
        else:
            run_top_candidate(subset, days_of_cover=DAYS_OF_COVER)


DISCOVERED CANDIDATES
  ISO3    Relevance                 Chokepoint                Spike   Brent 30d  Class
  ------  ------------------------  ----------------------  -------  ----------  ------------
  IRN     producer+chokepoint       Strait of Hormuz           174x      -28.2%  INVESTIGATE
  USA     producer                  -                          134x      -28.2%  INVESTIGATE
  CAN     producer                  -                           26x      -28.2%  INVESTIGATE
  YEM     producer+chokepoint       Bab el-Mandeb               26x      -28.2%  INVESTIGATE
  GBR     producer                  -                           10x      -28.2%  INVESTIGATE
  IRQ     producer                  -                            8x      -28.2%  INVESTIGATE
  SAU     producer                  -                            6x      -28.2%  INVESTIGATE
  VEN     producer                  -                            6x      -28.2%  INVESTIGATE

  To trace a specific candidate, change TRACE_TARGET

## Extensions (add when ready)

- **GDELT (events):** `pip install gdeltdoc` for light pulls, or query the
  `gdelt-bq.gdeltv2.events` public dataset in Google BigQuery with SQL — filter to a
  country/date/event-type and pull just that slice.
- **ACLED (verified events):** REST API with a key (free, non-commercial).
- **IMF PortWatch (chokepoint transits + Economy Monitor):** its datasets live on an
  ArcGIS Hub — each has a CSV download and a queryable REST endpoint (`.../query?where=...&f=json`).
- **OPEC MOMR (tables):** parse the monthly PDF with `pdfplumber` / `camelot` for the
  prices, demand, supply, stocks, and days-of-cover tables.
- **GPR Index (risk premium):** `pd.read_excel` from the Caldara–Iacoviello file.

**Refresh:** wrap the loaders in a scheduled run (cron, or a `schedule` loop) so the cache
refreshes daily; the `cached()` helper already skips re-fetching fresh data.

**Then:** `trace_event()` is where the data becomes a story — extend it to pull live prices
(§5), jet-fuel (IATA), and PortWatch transits so each of the five cascade layers is filled
from real data.
